# Crop vs band, as a controlled experiment

`kaggle_all_experiments` runs **one** standardisation policy per session, welded
to a corpus by `STREAM`. That welding is right for production — it makes a
band-mode bank over crop-mode data unspellable — and it is exactly what stops
that notebook answering *does crop beat band*. Its two streams differ in
corpus **and** policy **and** dihedral augmentation, so a difference between
their tables has three candidate causes.

This notebook holds everything else fixed and moves one thing:

| arm | `--canon-mode` | `--crop-side` | `--geometric` | isolates |
| --- | --- | --- | --- | --- |
| `band` | band | — | no | the frozen stream's policy |
| `crop` | crop | 200 | no | **standardisation**, vs `band` |
| `crop_geo` | crop | 200 | yes | **dihedral aug**, vs `crop` |

One manifest. One row set, written to disk and fingerprinted once, so all three
extractions provably cover the same images in the same order. One eval
subsample, drawn from a fixed seed. `band` vs `crop` is then a single-variable
comparison, and `crop` vs `crop_geo` is a second one.

**What it answers.** Whether the standardisation policy changes
`heldout_robust_tpr_at_1pct` — the §6.4 selection metric — and by how much,
with bootstrap intervals. Plus the CPU-only confound reading from
`gate_confounds.py`, run on all three arms before any GPU time.

**What it does not answer.** Whether `coco_crop`'s *corpus* is better than the
frozen one. That is a different variable and this notebook deliberately does
not move it.

**These banks are not shards of anything.** The row subset is written as its
own manifest with its own fingerprint, so `merge_banks` will refuse them and
should. They exist to be compared with each other and then deleted.


## 0. Parameters


In [ ]:
# ============ THE ONLY LINES YOU NORMALLY EDIT ============
STREAM   = "coco_crop"    # "frozen" | "coco_crop" -- the CORPUS, held fixed
BACKBONE = "siglip2l"
ARMS     = ["band", "crop", "crop_geo"]
N_ROWS   = 10000          # train+val_internal rows, stratified. See the cost cell.
VERIFY_SAMPLE = 5000      # rows digested by the verify gate. See section 3.
SMOKE    = True           # True first: proves the chain in minutes
PHASES   = "auto"         # "auto", or e.g. "gate1" / "stage_a,ladder"
# ==========================================================

# The corpus is now a plain choice of corpus, NOT a choice of policy: the
# policy is the thing under test and it comes from ARMS. That is the one
# deliberate difference from kaggle_all_experiments, where the two travel
# together so that a mismatch is unspellable. Here a "mismatch" is the
# experiment, so the guard is moved: every arm records its own policy in its
# own bank config, and the comparison cell refuses to tabulate two banks whose
# recorded policies are not the ones this notebook asked for.
STREAMS = {
    "frozen": dict(slug="techjam-aigc-train",
                   eval_slug="techjam-aigc-eval-manifest"),
    "coco_crop": dict(slug="techjam-aigc-train-coco-crop",
                      eval_slug="techjam-aigc-eval-manifest-coco-crop"),
}
assert STREAM in STREAMS, f"STREAM must be one of {sorted(STREAMS)}"
CFG = STREAMS[STREAM]

# The policy each arm extracts under. `geometric` is legal only under a square
# standardisation, so only under crop -- run_shard.py refuses it otherwise and
# kb.run_shard_argv refuses it a second earlier.
CROP_SIDE = 200
ARM_POLICY = {
    "band":     dict(canon_mode="band", crop_side=None, geometric=False),
    "crop":     dict(canon_mode="crop", crop_side=CROP_SIDE, geometric=False),
    "crop_geo": dict(canon_mode="crop", crop_side=CROP_SIDE, geometric=True),
}
assert set(ARMS) <= set(ARM_POLICY), f"unknown arm in {ARMS}"

# An eval bank must be extracted under the SAME canon mode as the training
# bank whose rung it scores -- a robustness curve measured on pixels the head
# never saw is not that head's curve. Dihedral is a TRAINING-side augmentation
# and is deliberately absent from the eval grid (the grid scores one image
# under 20 conditions; a per-condition reorientation would confound "the score
# fell under jpeg_q30" with "it was a different picture"). So `crop` and
# `crop_geo` share one crop-mode eval bank, and there are two eval banks for
# three arms, not three.
EVAL_MODE_OF = {a: ARM_POLICY[a]["canon_mode"] for a in ARMS}

RUNGS = ["a0", "a3"]   # a0 = the linear probe, the cleanest read of feature
                       # quality; a3 = the shipping system. Adding rungs is
                       # cheap once the bank is cached -- Stage B only.

# Rows per split in the eval subsample. heldout_generator is half of the §6.4
# selection population, so it is not starved relative to val_internal.
EVAL_SUBSAMPLE = {"val_internal": 2000, "heldout_generator": 2000,
                  "benchmark": 2000}

SPLITS = "train,val_internal"      # Stage B needs BOTH. Do not narrow this.
SEED   = 20260827                  # must match scripts/extract_features.py
TIER   = "ablation"

WORKERS          = 4
BATCH_SIZE       = 16
CHECKPOINT_EVERY = 200

#: Measured on a T4, siglip2l, 11 views, shard 0 of the frozen corpus:
#: 26,224 images in 2.60 h. Used only to print a cost estimate.
RATE_S_PER_IMAGE = 0.357

REPO_URL = "https://github.com/bersamin12/robust-aigc-detection"
BRANCH   = "feat/robust-aigc-detection"
REPO_DIR = "/kaggle/working/robust-aigc-detection"

MANIFEST_GLOB      = f"/kaggle/input/**/{CFG['slug']}/manifest.parquet"
DATA_GLOB          = f"/kaggle/input/**/{CFG['slug']}"
EVAL_MANIFEST_GLOB = f"/kaggle/input/**/{CFG['eval_slug']}*/eval_manifest*.parquet"
BENCH_MOUNT_GLOB   = "/kaggle/input/**/techjam-aigc-benchmark"

WORK     = "/kaggle/working"
ABL      = f"{WORK}/ablation"
SUB_MANIFEST = f"{ABL}/manifest_ablation.parquet"
BANK_DIR = lambda arm: f"{WORK}/banks/ablate_{BACKBONE}_{arm}"
EVAL_DIR = lambda mode: f"{WORK}/banks/ablate_eval_{BACKBONE}_{mode}"
DOCS_DIR = lambda arm: f"{ABL}/docs/{arm}"
RUNS_DIR = lambda arm: f"{ABL}/rungs/{arm}"

print(f"corpus={STREAM} ({CFG['slug']})  backbone={BACKBONE}")
print(f"arms={ARMS}")
for a in ARMS:
    p = ARM_POLICY[a]
    print(f"  {a:9s} canon={p['canon_mode']:4s} crop_side={p['crop_side']} "
          f"geometric={p['geometric']}  eval bank={EVAL_MODE_OF[a]}")
print(f"rows={N_ROWS}  rungs={RUNGS}  smoke={SMOKE}")


## 1. Get the code

Same as `kaggle_all_experiments`: public repo, shallow clone, `reset --hard` on
a re-run, and both `notebooks/` and `src/` on `sys.path` before the editable
install — an editable install registers through a `.pth` file that `site.py`
reads at interpreter start, so a kernel that was already running never sees it.


In [ ]:
import glob, importlib, os, subprocess, sys, time

def sh(argv, **kw):
    print("$", " ".join(str(a) for a in argv))
    return subprocess.run([str(a) for a in argv], check=True, **kw)

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH])
    sh(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])
else:
    sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])

for _sub in ("notebooks", "src"):
    _p = os.path.join(REPO_DIR, _sub)
    if _p not in sys.path:
        sys.path.insert(0, _p)
importlib.invalidate_caches()
import kaggle_bootstrap as kb
importlib.reload(kb)

sh(["git", "-C", REPO_DIR, "log", "--oneline", "-1"])
print("helper loaded from", kb.__file__)


## 2. Install — without losing Kaggle's torch

`pip install -e .` hands pip the `torch>=2.0` line from `pyproject.toml` and
invites it to resolve a torch built for a different CUDA than this machine's
drivers. The project goes in with `--no-deps`; everything else only if
genuinely missing. **If you ever see `torch` in the printed plan, stop.**


In [ ]:
def installed_version(dist):
    try:
        import importlib.metadata as im
        return im.version(dist)
    except Exception:
        return None

plan = kb.install_plan(os.path.join(REPO_DIR, "pyproject.toml"), REPO_DIR,
                       transformers_version=installed_version("transformers"))
print("pip plan:")
for cmd in plan:
    print("   ", " ".join(cmd))
assert not any(w.split("=")[0].split(">")[0] in ("torch", "torchvision", "triton")
               for cmd in plan for w in cmd), "STOP: the plan would touch torch"
for cmd in plan:
    sh(cmd, capture_output=True, text=True)
print("\ninstall done")

importlib.invalidate_caches()
from aigcdet.features.backbones import BACKBONES as _B
print("aigcdet importable:", len(_B), "backbones registered")


In [ ]:
import platform

torch_v, tf_v = installed_version("torch"), installed_version("transformers")
problems = kb.environment_problems(platform.python_version(), torch_v, tf_v)
print(f"python {platform.python_version()}  torch {torch_v}  transformers {tf_v}")
if problems:
    for p in problems:
        print("\nPROBLEM:", p)
    raise SystemExit("fix the above before continuing")

import torch
print("cuda available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), (
    "no GPU. Settings > Accelerator > GPU, then restart the session.")

from aigcdet.features.backbones import BACKBONES
MODEL_ID = BACKBONES[BACKBONE].hf_id
if kb.requires_hf_token(BACKBONE):
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
    except Exception:
        secrets = None
    token = kb.hf_token(secrets)
    for line in kb.hf_auth_advice(token, MODEL_ID, gated=True):
        print(line)
    if not token:
        raise SystemExit("no HuggingFace token -- see above")
    os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = token
else:
    print(f"{MODEL_ID} is public -- no token required")


## 3. Attach the data and verify it

Same gate as the main notebook, and for the same reason: `GATE` is a required
argument of everything downstream, so the verification cannot be skipped by
accident.

`VERIFY_SAMPLE` is a parameter rather than a buried constant because on this
mount digesting costs about 0.26 s/row — 2,000 rows took 524 s in a real
session — so the full 182,150 would be roughly 13 h, a whole session before any
GPU work. 5,000 rows is about 22 minutes and is strong evidence that the
attached Datasets are the ones the manifest was frozen against. It is evidence
and not proof, and `describe_gate` keeps saying so. Raise it for a run whose
bank you intend to keep and publish; this one's banks are meant to be compared
and then deleted.

The globs are recursive (`**`) because Kaggle mounts a Dataset either at
`/kaggle/input/<slug>` or at `/kaggle/input/datasets/<owner>/<slug>` depending
on how it was attached, and both shapes turn up in this project's sessions.


In [ ]:
import pandas as pd

def one(pattern, what):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert hits, f"no {what} attached (looked for {pattern})"
    assert len(hits) == 1, (
        f"{len(hits)} candidates for {what}: {hits}. Detach the ones this run "
        "is not using -- picking one silently is how a bank ends up describing "
        "a corpus it does not contain.")
    return hits[0]

MANIFEST  = one(MANIFEST_GLOB, "manifest Dataset")
DATA_MOUNT = one(DATA_GLOB, "image Dataset")
print("manifest:", MANIFEST)
print("mount:   ", DATA_MOUNT)

EXPECTED = kb.top_level_names(pd.read_parquet(MANIFEST, columns=["rel_path"]))
print("dataset root should contain:", sorted(EXPECTED))
UNIFIED = kb.unify_mounts([DATA_MOUNT], "/kaggle/temp/aigcdet_root", EXPECTED)
DATA_ROOT = UNIFIED.root
print("unified root:", DATA_ROOT, "->", sorted(os.listdir(DATA_ROOT))[:8])

t0 = time.time()
manifest, GATE = kb.open_verified_manifest(
    MANIFEST, DATA_ROOT,
    sample=200 if SMOKE else VERIFY_SAMPLE,
    check_extra=not UNIFIED.linked)
print(kb.describe_gate(GATE))
print(f"\nverified in {time.time() - t0:.0f}s")
print(manifest["split"].value_counts().to_string())


## 4. The row set — written once, fingerprinted once

This is what makes the comparison controlled, so it is worth being explicit
about the two things it does.

**It drops rows crop mode cannot standardise.** `canonicalise` in crop mode
takes a 200×200 window at native resolution and *raises* rather than upscaling
an image too small to fill it — deliberately, since upscaling to reach the
window would reintroduce the resampling signature the whole module exists to
remove. They are dropped from **every** arm, including `band`, which does not
need the filter: the arms must cover identical rows, or the comparison is
between two row sets as much as between two policies.

**This is why `STREAM` defaults to `coco_crop`.** Measured against the
published manifests, 2026-08-30:

| corpus | rows | short side < 200 | what they are |
| --- | --- | --- | --- |
| `coco_crop` | 182,150 | **0** (min observed exactly 200) | — |
| `frozen` | 138,116 | 1,308 (0.95%) | every one generated, every one wildfake |

`coco_crop`'s preset sets `min_short_side: 200`, so the filter is a no-op there
and the cell says so. On the frozen corpus it is not a no-op and not a neutral
one: 1,260 of frozen **shard 0's** 26,224 rows are below the band, and all
1,260 are BigGAN at exactly 128px. BigGAN is bimodal at two resolutions —
1,260 at 128px and 2,240 at 200px — so the filter deletes one entire mode and
moves shard 0's balance from 10,049/16,175 to 10,049/14,915. Both arms still
cover identical rows, so a `frozen` run here is internally valid; it is just
no longer quite the corpus the main ladder trains on. Say so if you quote it.

**The existing shard-0 band bank cannot serve as the `band` arm.** It was
extracted over all 26,224 rows, including the 1,260 crop cannot take, and a
bank is aligned to its manifest positionally — so an unfiltered band bank and a
filtered crop bank are not row-comparable. Every arm here is extracted fresh
against the manifest this cell writes.

**It subsamples stratified, not contiguously.** `--shard i/N` slices the
manifest by position, and this manifest is ordered by source — shard 3 of the
frozen corpus contains zero fakes. `stratified_subsample` balances classes
first and then (generator × source) strata within each class, which is what
§4.4a already does for the benchmark, and it is applied per split so
`train` and `val_internal` keep their own proportions.

The result is written as its own manifest and re-read to take its fingerprint.
Every arm is then launched with `--expect-manifest-sha256` set to that value,
so an arm extracted against anything else refuses to start rather than
producing a bank that is quietly not comparable.


In [ ]:
import numpy as np
from aigcdet.data.manifest import MANIFEST_COLUMNS, MANIFEST_IDENTITY_COLUMNS, read_manifest
from aigcdet.eval.grid import stratified_subsample
from aigcdet.features.bank import manifest_fingerprint

os.makedirs(ABL, exist_ok=True)

pool = kb.select_splits(manifest, SPLITS)
short = np.minimum(pool["width"].to_numpy(), pool["height"].to_numpy())
too_small = pool[short < CROP_SIDE]
if len(too_small):
    print(f"dropping {len(too_small)} row(s) with short side < {CROP_SIDE} -- "
          "crop mode cannot standardise them, so no arm may contain them")
    print(too_small["label"].value_counts().rename("by label").to_string())
    print(too_small["source"].value_counts().head().rename("by source").to_string())
else:
    print(f"no rows below short side {CROP_SIDE} -- this corpus's preset "
          "already enforces it, so the filter is a no-op")
pool = pool[short >= CROP_SIDE]

# Per split, so train and val_internal each keep their own class balance
# rather than the larger split swallowing the budget.
parts = []
for split_name, part in pool.groupby("split", sort=True):
    quota = max(1, round(N_ROWS * len(part) / len(pool)))
    take = stratified_subsample(part.reset_index(drop=True), quota, seed=SEED)
    parts.append(part.iloc[take])
sub = pd.concat(parts).sort_index()

print(f"\nsubsample: {len(sub)} of {len(pool)} eligible rows")
print(sub.groupby(["split", "label"]).size().rename("rows").to_string())
assert sub["label"].nunique() == 2, "one class only -- raise N_ROWS"
for s in ("train", "val_internal"):
    assert (sub["split"] == s).any(), f"no {s} rows survived; raise N_ROWS"

# Written directly rather than through `write_manifest`, which would recompute
# every digest from the files -- a second full verification pass, for identity
# columns this frame already carries unchanged from the frozen manifest.
cols = MANIFEST_COLUMNS + [c for c in MANIFEST_IDENTITY_COLUMNS if c in sub.columns]
sub[cols].to_parquet(SUB_MANIFEST, index=False)

SUB_SHA = manifest_fingerprint(read_manifest(SUB_MANIFEST, root=DATA_ROOT))
SUB_GATE = kb.carried_gate(SUB_SHA, DATA_ROOT)
print(f"\nablation manifest: {SUB_MANIFEST}")
print(f"fingerprint {SUB_SHA[:16]}...  -- every arm is launched against this")

need_gb = kb.bank_bytes(len(sub), BACKBONES[BACKBONE].dim, n_views=11) / 1024**3
hours = len(sub) * RATE_S_PER_IMAGE / 3600
print(f"\nper arm: {need_gb:.2f} GiB, ~{hours:.1f} h at {RATE_S_PER_IMAGE} s/image")
print(f"{len(ARMS)} arm(s) -> ~{hours * len(ARMS):.1f} h of Stage A, "
      f"{need_gb * len(ARMS):.2f} GiB")
print(f"plus {len(set(EVAL_MODE_OF.values()))} eval bank(s) at "
      f"{sum(EVAL_SUBSAMPLE.values())} rows x 20 conditions "
      f"(~{sum(EVAL_SUBSAMPLE.values()) * 20 * RATE_S_PER_IMAGE / 11 / 3600:.1f} h each)")


## 5. Gate 1 — the confound reading, on the CPU, before any GPU time

`gate_confounds.py` decodes a stratified sample and runs it through the
pipeline's real front half — `canonicalise` under the arm's policy, then
`dihedral`, then `proxy_vector` — and reports orientation-corrected AUC per
low-level proxy. It draws view 0's *actual* window from the same per-view key
extraction would use, not a centre crop that merely resembles it.

This is half the answer and it costs minutes. `docs/dataset_presets.md` records
the same measurement on the built `coco_crop` corpus: worst confound 0.7038
under band against 0.6774 under crop+dihedral, with `laplacian_var` inside
WildFake falling 0.146. If this cell reproduces that ordering on your corpus,
the extraction is worth paying for; if band already wins here, say so and stop.

A rise in `worst` is not by itself a reason to abandon an arm — it is a reason
to read that arm's headline against `content_blind_auc` and
`stratified_auc.py --stratify-by source`, because crop trades a *spectral*
confound for a *content* one and these three proxies cannot see the second.


In [ ]:
GATE1 = f"{ABL}/gate1.json"

def _phase_done():
    import json
    return {
        "gate1": os.path.exists(GATE1),
        "stage_a": all(_bank_done(BANK_DIR(a)) for a in ARMS),
        "eval_bank": all(_bank_done(EVAL_DIR(m)) for m in set(EVAL_MODE_OF.values())),
        "ladder": all(os.path.exists(f"{DOCS_DIR(a)}/selection.json") for a in ARMS),
    }

def _bank_done(d):
    st = kb.read_resume_state(d)
    return st.exists and st.n_images > 0 and st.n_done >= st.n_images

ORDER = ["gate1", "stage_a", "eval_bank", "ladder"]
STATUS = _phase_done()
TODO = ([p for p in ORDER if not STATUS[p]] if PHASES == "auto"
        else [p.strip() for p in PHASES.split(",") if p.strip()])
for p in ORDER:
    print(f"  {p:10s} {'done' if STATUS[p] else ('TODO' if p in TODO else 'skip')}")
print(f"\nthis session will run: {TODO or '(nothing)'}")


In [ ]:
import json


def parse_pooled(lines):
    """The POOLED row of gate_confounds' table, as {proxy: auc}.

    Parsed off the header rather than by column position: the row begins with
    `group n real fake` and only then the proxies, and `PROXY_NAMES` is free
    to grow. A header-driven parse gains a column silently; a positional one
    would mislabel every value after it.
    """
    header = None
    for line in lines:
        toks = line.split()
        if not toks:
            continue
        if toks[0] == "group":
            header = toks
        elif header and toks[0] == "POOLED" and len(toks) == len(header):
            out = {}
            for name, val in zip(header[4:], toks[4:]):
                try:
                    out[name] = float(val)
                except ValueError:
                    pass
            return out
    return {}


if "gate1" in TODO:
    rows = {}
    for arm in ARMS:
        p = ARM_POLICY[arm]
        argv = [sys.executable, f"{REPO_DIR}/scripts/gate_confounds.py",
                "--manifest", SUB_MANIFEST, "--split", SPLITS,
                "--canon-mode", p["canon_mode"],
                "--n", "600" if SMOKE else "6000",
                "--seed", str(SEED)]
        if p["crop_side"] is not None:
            argv += ["--crop-side", str(p["crop_side"])]
        if p["geometric"]:
            argv.append("--geometric")
        print(f"\n===== {arm} " + "=" * 40)
        # AIGCDET_DATA_ROOT because gate_confounds takes no --root: read_manifest
        # falls back to it, which is what that variable exists for.
        env = dict(os.environ, AIGCDET_DATA_ROOT=DATA_ROOT)
        lines = []
        rc = kb.run_streaming(argv, on_line=lambda s: (print(s), lines.append(s)),
                              env=env)
        assert rc == 0, f"gate_confounds for {arm} exited {rc}"
        rows[arm] = parse_pooled(lines)
        assert rows[arm], (
            f"could not find a POOLED row in {arm}'s output. The script ran "
            "and exited 0, so read its table above by eye rather than trusting "
            "this cell's comparison.")
    with open(GATE1, "w") as f:
        json.dump(rows, f, indent=2)
else:
    print("gate1: skipped")

if os.path.exists(GATE1):
    g1 = pd.DataFrame(json.load(open(GATE1)))
    print("\nGate 1 -- orientation-corrected AUC per proxy (lower is better)\n")
    print(g1.to_string(float_format=lambda v: f"{v:.4f}"))
    if {"band", "crop"} <= set(g1.columns):
        print("\ncrop - band:")
        print((g1["crop"] - g1["band"]).to_string(
            float_format=lambda v: f"{v:+.4f}"))


## 6. Stage A — one bank per arm

Same `run_shard.py` the fleet uses, with `--shard 0 --n-shards 1` because the
row set was already chosen in section 4. Each arm resumes independently, so a
session that dies in arm 2 costs at most `CHECKPOINT_EVERY` images and the
finished arms are untouched.

`kb.run_shard_argv` refuses `geometric=True` under band mode before the shard
is even resolved, and `run_shard.py` refuses it again after — a 90° rotation
transposes a non-square image and every op downstream is shape-preserving.


In [ ]:
if "stage_a" in TODO:
    for arm in ARMS:
        out = BANK_DIR(arm)
        if _bank_done(out):
            print(f"{arm}: already complete, skipping")
            continue
        p = ARM_POLICY[arm]
        argv = kb.run_shard_argv(
            SUB_GATE, manifest_path=SUB_MANIFEST, root=DATA_ROOT,
            backbone=BACKBONE, out_dir=out, splits=SPLITS, shard=0, n_shards=1,
            resume=True, workers=WORKERS, batch_size=BATCH_SIZE,
            checkpoint_every=CHECKPOINT_EVERY, limit=64 if SMOKE else None,
            **p)
        print(f"\n===== {arm} " + "=" * 40)
        print(" ".join(argv[1:]), "\n")
        t0 = time.time()
        rc = kb.run_streaming(argv)
        assert rc == 0, f"stage A for {arm} exited {rc}"
        print(f"\n{arm} finished in {(time.time() - t0)/60:.1f} min")
else:
    print("stage_a: skipped")

for arm in ARMS:
    st = kb.read_resume_state(BANK_DIR(arm))
    if st.exists:
        print(f"  {arm:9s} {st.n_done}/{st.n_images} ({st.fraction_done:.0%})")


## 7. Eval banks — one per canon mode, not one per arm

Two banks for three arms: `crop` and `crop_geo` are scored against the same
crop-mode eval bank, because dihedral is a training-side augmentation and the
eval grid is deliberately without it.

The eval manifest is rooted one level **above** the training root — its
`rel_path`s start either with `demo/` (the organisers' benchmark) or with the
normalised tree's own name — so `DATA_ROOT`, which is the inside of the latter,
resolves nothing on its own. A two-link farm is built instead, and 200 sampled
rows are checked to actually resolve through it before an hour of GPU: a farm
can list correctly and resolve to nothing when a mount arrived under a
different slug.

The subsample is drawn by `stratified_subsample` from a fixed seed, so both
eval banks cover the same images.


In [ ]:
import shutil

if "eval_bank" in TODO:
    EVAL_MANIFEST = one(EVAL_MANIFEST_GLOB, "eval manifest Dataset")
    BENCH_MOUNT = one(BENCH_MOUNT_GLOB, "benchmark Dataset")
    print("eval manifest:", EVAL_MANIFEST)
    print("benchmark:    ", BENCH_MOUNT)

    rel = pd.read_parquet(EVAL_MANIFEST, columns=["rel_path"])["rel_path"]
    expected = sorted({x.split("/")[0] for x in rel})
    train_names = [n for n in expected if n != "demo"]
    assert len(train_names) == 1 and "demo" in expected, (
        f"eval manifest top-level names are {expected}; this cell knows how to "
        "link exactly 'demo' plus one normalised tree.")

    EVAL_ROOT = "/kaggle/temp/aigcdet_eval_root"
    shutil.rmtree(EVAL_ROOT, ignore_errors=True)
    os.makedirs(EVAL_ROOT, exist_ok=True)
    os.symlink(DATA_ROOT, os.path.join(EVAL_ROOT, train_names[0]))
    os.symlink(BENCH_MOUNT, os.path.join(EVAL_ROOT, "demo"))
    missing = [x for x in rel.sample(200, random_state=SEED)
               if not os.path.exists(os.path.join(EVAL_ROOT, x))]
    assert not missing, (
        f"{len(missing)} of 200 sampled rows do not resolve, e.g. {missing[:3]}")
    print("eval root:", EVAL_ROOT, "-- 200 sampled rows all resolve")

    for mode in sorted(set(EVAL_MODE_OF.values())):
        out = EVAL_DIR(mode)
        if _bank_done(out):
            print(f"eval {mode}: already complete, skipping")
            continue
        argv = [sys.executable, f"{REPO_DIR}/scripts/extract_eval_bank.py",
                "--manifest", EVAL_MANIFEST, "--backbone", BACKBONE,
                "--out", out, "--tier", TIER, "--root", EVAL_ROOT,
                "--device", "cuda", "--batch-size", str(BATCH_SIZE),
                "--checkpoint-every", str(CHECKPOINT_EVERY), "--resume",
                "--canon-mode", mode, "--subsample-seed", str(SEED)]
        for split_name, n in EVAL_SUBSAMPLE.items():
            argv += ["--subsample", f"{split_name}={n}"]
        if mode == "crop":
            argv += ["--crop-side", str(CROP_SIDE)]
        if SMOKE:
            argv += ["--limit", "64"]
        print(f"\n===== eval {mode} " + "=" * 35)
        print(" ".join(argv[1:]), "\n")
        rc = kb.run_streaming(argv)
        assert rc == 0, f"eval bank {mode} exited {rc}"
else:
    print("eval_bank: skipped")


## 8. The ladder, once per arm

`run_ablation.py` trains each rung, scores it against that arm's eval bank,
writes the robustness table and applies §6.4's selection rule. It is itself
resumable: a rung whose checkpoint matches its config is not retrained.

Stage B reads cached features, so this is minutes per rung, not hours — adding
rungs to `RUNGS` after the banks exist is cheap.

**Under `SMOKE` this cell will fail, and correctly.** A 64-row eval bank holds
only `val_internal`, and §6.4's selection population needs
`heldout_generator` too, so `check_selection_population` refuses it. The smoke
run's job is to prove the chain up to here; set `SMOKE = False` for a table.


In [ ]:
if "ladder" in TODO:
    for arm in ARMS:
        os.makedirs(DOCS_DIR(arm), exist_ok=True)
        cfgs = [f"{REPO_DIR}/configs/rungs/{r}.yaml" for r in RUNGS]
        for c in cfgs:
            assert os.path.exists(c), f"missing rung config {c}"
        argv = [sys.executable, f"{REPO_DIR}/scripts/run_ablation.py",
                "--bank", BANK_DIR(arm),
                "--eval-bank", EVAL_DIR(EVAL_MODE_OF[arm]),
                "--rungs", *cfgs, "--tier", TIER, "--device", "cuda",
                "--out", f"{DOCS_DIR(arm)}/robustness_table.md",
                "--selection", f"{DOCS_DIR(arm)}/selection.json",
                "--heatmap", f"{DOCS_DIR(arm)}/robustness_heatmap.png",
                "--out-dir", RUNS_DIR(arm)]
        print(f"\n===== {arm} " + "=" * 40)
        print(" ".join(argv[1:]), "\n")
        rc = kb.run_streaming(argv)
        assert rc == 0, f"ladder for {arm} exited {rc}"
else:
    print("ladder: skipped")


## 9. The comparison

`heldout_robust_tpr_at_1pct` on the §6.4 selection population, per rung, per
arm. This is the number the project selects on, so it is the one the policy
question has to be settled against.

Before tabulating, each bank's recorded `canon` config is checked against the
policy this notebook asked its arm to use. `BankWriter` treats every
unrecognised config key as must-match, so a crop bank and a band bank are
distinguishable on disk — but only if someone looks, and a table built from two
banks that are secretly the same policy would look exactly like a null result.


In [ ]:
rows, policies = {}, {}
for arm in ARMS:
    sel_path = f"{DOCS_DIR(arm)}/selection.json"
    if not os.path.exists(sel_path):
        print(f"{arm}: no selection.json yet")
        continue
    cfg = json.load(open(f"{BANK_DIR(arm)}/config.json"))
    # `canon_policy`, which is what `CanonPolicy.as_record()` is filed under in
    # the bank config -- not `canon`. Read the key wrong and this cell raises
    # AFTER every hour of extraction and training has already succeeded.
    got = cfg.get("canon_policy")
    want = ARM_POLICY[arm]
    assert got, (
        f"the bank for {arm} records no canon policy at all -- it predates "
        "CanonPolicy and cannot be attributed to a standardisation")
    assert got.get("mode") == want["canon_mode"], (
        f"arm {arm!r} asked for mode={want['canon_mode']!r} but its bank "
        f"records {got!r}. These are not the arms you think they are.")
    policies[arm] = got
    sel = json.load(open(sel_path))
    rows[arm] = {r: v.get("heldout_robust_tpr_at_1pct")
                 for r, v in sel.get("summary", {}).items()}
    print(f"{arm:9s} canon={got}  headline={sel.get('headline')}")

if rows:
    table = pd.DataFrame(rows).reindex(columns=[a for a in ARMS if a in rows])
    print("\nheldout_robust_tpr_at_1pct (higher is better)\n")
    print(table.to_string(float_format=lambda v: f"{v:.4f}"))
    if {"band", "crop"} <= set(table.columns):
        print("\ncrop - band  (standardisation):")
        print((table["crop"] - table["band"]).to_string(
            float_format=lambda v: f"{v:+.4f}"))
    if {"crop", "crop_geo"} <= set(table.columns):
        print("\ncrop_geo - crop  (dihedral augmentation):")
        print((table["crop_geo"] - table["crop"]).to_string(
            float_format=lambda v: f"{v:+.4f}"))
    table.to_csv(f"{ABL}/comparison.csv")
    print(f"\nwritten: {ABL}/comparison.csv")
    print("per-arm tables:")
    for arm in rows:
        print(f"  {DOCS_DIR(arm)}/robustness_table.md")


## 10. How to read this, and how not to

**The interval matters more than the sign.** With `EVAL_SUBSAMPLE` at 2,000
rows per split, a TPR at 1% FPR is estimated from roughly twenty false
positives' worth of threshold. Each arm's own `robustness_table.md` carries
the bootstrap intervals `run_ablation` computed; read the difference against
them before calling a winner. If the intervals overlap, the honest answer is
that this row budget did not separate the policies — raise `N_ROWS` and
`EVAL_SUBSAMPLE`, or accept the null.

**`crop` beating `band` on this metric is not the whole case for crop.** Crop
trades a spectral confound for a content one: a 200×200 window is a whole
frame for a 200px image and a detail for a 640px photograph, so field of view
becomes correlated with native resolution. The three Gate 1 proxies cannot see
that. Before quoting a crop headline anywhere, run the two controls that can:

```bash
python scripts/stratified_auc.py --stratify-by source \
    --checkpoint <rung> --bank <the crop bank> --manifest <SUB_MANIFEST>
```

and `eval/controls.py:metadata_control`, which must sit at chance — crop hands
the backbone the same size for every image by construction, so anything else
means a size cue survived a transform designed to remove it.

**This is one corpus.** The result is about the policy *on this corpus*, and
`docs/dataset_presets.md` shows the two interact: band mode fails a
`--max-auc 0.70` gate on `coco_crop` while passing on the frozen corpus, because
swapping WildFake's soft 200px reals for photographs makes the sharpness
shortcut easier. Running this notebook under both `STREAM` values answers a
larger question than either run does alone.

**Do not merge these banks.** They were extracted against
`manifest_ablation.parquet`, which has its own fingerprint and its own row
order, so `merge_banks` will refuse them and `assert_fusion_parents` will
refuse to fuse them at A5. That refusal is correct. Publish
`comparison.csv` and the per-arm tables; delete the banks.
